### This notebook assign the districts for the 27 basins of IKI Project for the indicators using a proportionate method. 

**Created:** 6/10/2025 by Jorge Mayo (jmayo@rti.org)  
**Project #:** 0219481  
**Last modified:**
**Status:** 
**QA Status:** reviewed by  
**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Exposicion\Scripts_Exposure
 
**Objective:**   
**Compatibility:** 
**Packages:** numpy, pandas, geopandas, sqlite3, matplotlib ...  
**Further documentation:**  
 
**Inputs:**   
**Outputs:** COMID_District_Fractions.csv (to be used in function)
 
**Assumptions:** assumptions on script operations (e.g., inputs in subfolder of wd) or on calculations or theories used in code
 
**Future work:** 
 
**Notes:** 

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt

In [3]:
#user = 'jmayo'
#user= 'cpickering'
user = 'sgilson'

In [ ]:
### Reading the AHD shapefile for the 27 basins and projecting to UTM Zone 18S (EPSG:32718)
subbasins_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_catchment_modeling.shp'
subbasins_gdf = gpd.read_file(subbasins_shapefile).to_crs('EPSG:32718')
subbasins_gdf.drop(columns=['Obser'], inplace=True)

### Reading the district shapefile for Peru (1885 total)
districts_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/GIS/5_ADMIN_BOUNDARIES_VIAS_ROADS_CIUDADES_CITIES/Limite_Demografico/Shp_Distritos_2021/ngeo_Distritos_2021.shp'
districts_gdf = gpd.read_file(districts_shapefile).to_crs('EPSG:32718')
#districts_gdf

## Find the Proportion of the district within COMID method

Note: turn this into a function that finds the proportion for any polgyon input

In [12]:
# 1. Intersect subbasins and districts to get overlapping polygons
intersections = gpd.overlay(subbasins_gdf, districts_gdf[['IDDIST', 'NOMBDIST', 'geometry']], how='intersection')

# 2. Calculate area of each intersection polygon (in CRS units, usually meters)
intersections['area_within_comid'] = intersections.geometry.area

# 3. Calculate total area for each district
districts_gdf['total_district_area'] = districts_gdf.geometry.area

# 4. Merge total district area into intersections
intersections = intersections.merge(
    districts_gdf[['IDDIST', 'total_district_area']], 
    on='IDDIST', 
    how='left'
)

# 5. Calculate the fraction of district area within each COMID
intersections['district_area_within_COMID_fraction'] = (
    intersections['area_within_comid'] / intersections['total_district_area']
)

# 6. Create the final dataframe with the required columns
result_df = intersections[['COMID', 'IDDIST', 'district_area_within_COMID_fraction']].copy()
result_df = result_df.rename(columns={'IDDIST': 'DISTRICT'})

# Display the result
print("Sample of results:")
print(result_df.head(10))

print(f"\nTotal number of COMID-District combinations: {len(result_df)}")
print(f"Number of unique COMIDs: {result_df['COMID'].nunique()}")
print(f"Number of unique Districts: {result_df['DISTRICT'].nunique()}")

# Optional: Save to CSV
result_df.to_csv(f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2a_Metodologia/Indicadores/COMID_District_Fractions.csv', index=False)

Sample of results:
       COMID DISTRICT  district_area_within_COMID_fraction
0  311153600   210102                             0.029839
1  311153600   210112                             0.275357
2  311153600   210106                             0.026153
3  310832400   210607                             0.016200
4  310832400   210210                             0.343789
5  310832400   210205                             0.219038
6  310832400   210605                             0.407705
7  310832400   211105                             0.083932
8  310825700   210703                             0.090383
9  310825700   210205                             0.079817

Total number of COMID-District combinations: 10204
Number of unique COMIDs: 3563
Number of unique Districts: 1299


### Assign the majority district area to each COMID (one to one)

In [ ]:
# 1. Intersect subbasins and districts to get overlapping polygons
intersections = gpd.overlay(subbasins_gdf, districts_gdf[['IDDIST', 'NOMBDIST', 'geometry']], how='intersection')

# 2. Calculate area of each intersection polygon (in CRS units, usually meters)
intersections['area_m2'] = intersections.geometry.area

# 3. Find the district with the largest intersection area per subbasin
idx = intersections.groupby('COMID')['area_m2'].idxmax()
largest_overlap = intersections.loc[idx]

# 4. Merge district info back to the original subbasins (keep all original geometry)
subbasins_with_district = subbasins_gdf.merge(
    largest_overlap[['COMID', 'IDDIST', 'NOMBDIST']],
    on='COMID',
    how='left'
)

subbasins_with_district.to_file(
    f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp',
    driver='ESRI Shapefile'
)

# Find COMIDs present in subbasins_gdf but missing in subbasins_with_district
dropped_comids = set(subbasins_gdf['COMID']) - set(subbasins_with_district['COMID'])
print(f"Number of dropped COMIDs: {len(dropped_comids)}")
print("Dropped COMIDs:", dropped_comids)

C:\Users\jmayo\AppData\Local\Temp\ipykernel_32520\711604970.py:2: UserWarning: `keep_geom_type=True` in overlay resulted in 2 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  intersections = gpd.overlay(subbasins_gdf, districts_gdf[['IDDIST', 'NOMBDIST', 'geometry']], how='intersection')
